# Filtered & multi-tenant RAG: vector search + filters in one query

Real retrieval rarely means "just find similar text." You need **semantic relevance under hard constraints** — under this price, above this rating, mentioning this keyword, belonging to this tenant.

With [Infino](https://pypi.org/project/infino/) the vectors and the metadata live in **one table**, so you combine vector k-NN with filters with no separate metadata store and no per-tenant namespaces. This notebook shows two complementary filtering paths on a real **Amazon product catalog**:

1. **Scalar filters** (price, rating, category) via a SQL `WHERE` over the `vector_search` table function.
2. **Keyword-constrained search** via a **pushdown pre-filter** — the k-NN ranks only among rows whose text matches, so a selective filter never starves the result.

## Setup

In [1]:
# %pip install infino pyarrow sentence-transformers datasets

In [2]:
import sys
from pathlib import Path

# Make the repo's shared helpers importable.
sys.path.insert(0, str(Path.cwd().parent))

import shutil

import infino
import pyarrow as pa

from _shared.embedding import DIM, METRIC, embed, embed_query, as_vector_column
from _shared.loaders import load_amazon
from _shared.sql import sql_lit, query

## 1. Load a product catalog

Each product has a title + description (what we search) plus metadata: price, average rating, category, and store/brand.

In [3]:
products = load_amazon(n=1200)
print(f"loaded {len(products)} products")
p = products[0]
print(p["title"])
print(f"  price=${p['price']}  rating={p['rating']}  category={p['category']!r}  store={p['store']!r}")

loaded 1200 products
GiGi Professional Multi-Purpose Wax Warmer, with See-Through Cover, White
  price=$25.87  rating=4.6  category='All Beauty'  store='GiGi'


## 2. Index text, vectors, and metadata in one table

The metadata columns are stored and SQL-filterable; `text` is full-text indexed and `emb` is vector indexed — all over the same rows.

In [4]:
DB_DIR = "./amazon_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("title", pa.large_utf8(), nullable=False),
    pa.field("text", pa.large_utf8(), nullable=False),
    pa.field("price", pa.float64(), nullable=False),
    pa.field("rating", pa.float64(), nullable=False),
    pa.field("category", pa.large_utf8(), nullable=False),
    pa.field("store", pa.large_utf8(), nullable=False),
    pa.field("emb", pa.list_(pa.float32(), DIM), nullable=False),
])
spec = infino.IndexSpec().fts("text").vector("emb", DIM, n_cent=256, metric=METRIC)
table = db.create_table("products", schema, spec)

embeddings = embed([p["text"] for p in products])


def column(key, ty):
    return pa.array([p[key] for p in products], type=ty)


table.append(pa.record_batch([
    column("title", pa.large_utf8()), column("text", pa.large_utf8()),
    column("price", pa.float64()), column("rating", pa.float64()),
    column("category", pa.large_utf8()), column("store", pa.large_utf8()),
    as_vector_column(embeddings),
], schema=schema))
print("indexed", db.query_sql("SELECT COUNT(*) AS n FROM products").to_pydict()["n"][0], "products")

indexed 1200 products


## 3. Scalar filters: vector search + SQL `WHERE`

The `vector_search` table function returns the nearest rows **with their columns**, so a `WHERE` on `price`/`rating` filters them directly — one statement, one table, no join.

A scalar predicate is applied after the k-NN ranks, so when it is selective, ask for more candidates (a larger k) and keep the top few that pass.

In [5]:
question = "hydrating moisturizer for dry skin"
qvec = ",".join(str(x) for x in embed_query(question))

budget = query(
    db,
    f"SELECT title, price, rating, score "
    f"FROM vector_search('products', 'emb', '{qvec}', 300) "
    f"WHERE price < 25 AND rating >= 4.5 "
    f"ORDER BY score ASC LIMIT 5"
)
print("Budget picks (< $25, rating >= 4.5):")
for title, price, rating in zip(budget["title"], budget["price"], budget["rating"]):
    print(f"  ${price:<6.2f} {rating}\u2605  {title[:75]}")

Budget picks (< $25, rating >= 4.5):
  $16.02  4.8★  Aquaphor Healing Ointment - Moisturizing Skin Protectant for Dry Cracked Ha
  $4.71   4.5★  Garnier SkinActive Moisture Bomb The Super Hydrating Smoothing Mask, 1.08 F
  $9.79   4.5★  Palmers Cocoa Butter Formula With Vitamin E 24 Hour Moisture (25% Bonus) 12
  $6.28   4.7★  Johnson's Skin Nourishing Moisture Baby Body Wash with Aloe Scent & Vitamin
  $16.92  4.6★  CeraVe Hydrating Oil Cleanser 236ml


Same query, different constraints — just change the `WHERE`. No reindexing, no second system.

In [6]:
premium = query(
    db,
    f"SELECT title, price, score FROM vector_search('products', 'emb', '{qvec}', 300) "
    f"WHERE price >= 25 ORDER BY score ASC LIMIT 5"
)
print("Premium (>= $25):")
for title, price in zip(premium.get("title", []), premium.get("price", [])):
    print(f"  ${price:<7.2f} {title[:75]}")

Premium (>= $25):
  $62.12   Mckesson Moisturizer Mckesson 18 Oz. Pump Bottle (#53-28007-18, Sold Per Ca
  $63.96   Jones Road Miracle Balm Au Naturel - 1.76 oz Citrus Scented Moisturizer for
  $79.99   Designer Skin Angel, 20-Ounce Bottle
  $31.44   Colonial Dames Concentrated Vitamin E Moisturizing Cream 42,000 I.U. for Hy
  $26.95   Raw African Shea Butter 32 oz. / 2 lbs. Jar (2 pack) Bulk Wholesale 100% Pu


## 4. Keyword-constrained search (pushdown pre-filter)

Sometimes the constraint is a **keyword the result must contain** — "semantically similar, but only products that mention *organic*." Pass `filter_column` + `filter_query` to `vector_search`: the k-NN ranks **only among rows whose text matches**, applied *during* the index scan.

This is a true pushdown pre-filter, not a post-filter — so even a highly selective keyword returns the k nearest *matching* rows, instead of filtering a fixed top-k that might leave you with nothing.

In [7]:
qv = embed_query(question)

# Unconstrained: nearest products overall.
plain = table.vector_search("emb", qv, k=5, projection=["title", "score"]).to_pydict()
print("Nearest overall:")
for title in plain["title"]:
    print("  -", title[:75])

# Pushdown: nearest products whose text mentions 'organic'.
organic = table.vector_search(
    "emb", qv, k=5,
    filter_column="text", filter_query="organic", filter_mode="or",
    projection=["title", "score"],
).to_pydict()
print("\nNearest that mention 'organic':")
for title in organic.get("title", []):
    print("  -", title[:75])

Nearest overall:
  - Aquaphor Healing Ointment - Moisturizing Skin Protectant for Dry Cracked Ha
  - Garnier SkinActive Moisture Bomb The Super Hydrating Smoothing Mask, 1.08 F
  - Palmers Cocoa Butter Formula With Vitamin E 24 Hour Moisture (25% Bonus) 12
  - Mckesson Moisturizer Mckesson 18 Oz. Pump Bottle (#53-28007-18, Sold Per Ca
  - Vaseline Clinical Care Body Cream Extremely Dry Skin Rescue 7.1 oz

Nearest that mention 'organic':
  - Mommy's Bliss Organic Gripe Water Gel for Newborns, Extra Gentle Gel, Relie
  - Under Eye Serum
  - Wellements Certified USDA Organic Liquid Probiotic 4 Fl Oz, Gripe Water & P
  - Babo Botanicals Soothing Baby Diaper Cream - with Non-Nano Zinc Oxide, Coll
  - Raw Shea Butter by Shea Moisture Extra Moisture Retention Shampoo 379ml


## 5. Multi-tenant isolation

Multi-tenant retrieval is just another filter. Here each catalog section (`category`) is a tenant: the **same** query, scoped to one tenant's rows, never leaks results from another — one shared table, no per-tenant index.

In [8]:
n_rows = db.query_sql("SELECT COUNT(*) AS n FROM products").to_pydict()["n"][0]
tenants = db.query_sql(
    "SELECT category, COUNT(*) c FROM products GROUP BY category ORDER BY c DESC LIMIT 2"
).to_pydict()["category"]

for tenant in tenants:
    rows = query(
        db,
        f"SELECT title, category, score FROM vector_search('products', 'emb', '{qvec}', {n_rows}) "
        f"WHERE category = {sql_lit(tenant)} ORDER BY score ASC LIMIT 3"
    )
    print(f"\ntenant = {tenant!r}")
    for title in rows.get("title", []):
        print("  -", title[:75])


tenant = 'All Beauty'
  - Aquaphor Healing Ointment - Moisturizing Skin Protectant for Dry Cracked Ha
  - Garnier SkinActive Moisture Bomb The Super Hydrating Smoothing Mask, 1.08 F
  - Palmers Cocoa Butter Formula With Vitamin E 24 Hour Moisture (25% Bonus) 12

tenant = 'Health & Personal Care'
  - Johnson's Skin Nourishing Moisture Baby Body Wash with Aloe Scent & Vitamin
  - Aquaphor Baby Healing Ointment Advanced Therapy Skin Protectant, Dry Skin a
  - hydraSense Nasal Aspirator Starter Kit


## What just happened

Vector relevance and constraints resolved together over one Infino table, two ways: scalar `WHERE` filters (price, rating, tenant) on the `vector_search` table function, and a **keyword pushdown pre-filter** that ranks only among matching rows. No separate metadata database, no namespace plumbing — the filter is just a column or a keyword.

**Next:** [`04_chat_rag.ipynb`](04_chat_rag.ipynb) ties retrieval and hybrid search into a multi-turn chatbot over your documents.